# Phase 4c: Feature-Covered Sample Synthesis (Round 2)

Αυτό το notebook αναλαμβάνει το **δεύτερο round** synthesis του Phase 4. Διαβάζει τα contrastive pairs που παρήγαγε ο pipeline 4a→4b (JSONL αρχείο) και χρησιμοποιεί το **Uncensored Llama-3.1-8B-Instruct-abliterated** για να δημιουργήσει **βελτιωμένα** τοξικά queries, αυτή τη φορά καθοδηγούμενα από τα SAE activation scores.

---

### Διαφορές από Phase 4a

| | **Phase 4a** (Round 1) | **Phase 4c** (Round 2) |
|---|---|---|
| **Script** | `generate_data_llama_r1.py` | `generate_data_llama_r2.py` |
| **Φάκελος** | `step1_contrastive_pair_construction/` | `step2_feature_covered_sample_synthesis/` |
| **Input Format** | TSV (FeatureID \t Summary \t Words) | **JSONL** (ένα JSON object ανά γραμμή, με contrastive pairs) |
| **Prompt** | Μόνο Feature Summary + Example Spans | Feature Summary + **Good Example** + **Bad Example** + **Activation Scores** + **Activated Spans** |
| **Μοντέλο** | `mlabonne/...abliterated` | `mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated` (uncensored) |
| **Few-shot Examples** | 3 απλά παραδείγματα | 3 μεγάλα παραδείγματα με contrastive context |
| **Πλήθος ανά feature** | 2 queries (configurable) | 2 queries (num_return_sequences=2) |
| **Resume Support** | ✅ progress.txt file | ✅ progress.txt file (Υποστηρίζεται πλέον πλήρως!) |

---

### ⚠️ ΣΗΜΑΝΤΙΚΟ: Hugging Face Token
Αν δεν έχεις ήδη ρυθμίσει HF_TOKEN στα Colab Secrets, ακολούθησε τα βήματα από το Phase 4a notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Επιβεβαίωση ότι έχουμε GPU (T4 ή L4)
!nvidia-smi

## 1. Εγκατάσταση Βιβλιοθηκών

In [ ]:
!pip install -q transformers==4.43.4 accelerate==0.33.0 bitsandbytes datasets

## 2. Σύνδεση με Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(hf_token)

## 3. Λήψη Κώδικα (FAC-Synthesis)

In [ ]:
%%bash
if [ ! -d "/content/FAC-Synthesis" ]; then
  git clone https://github.com/michalispsy/SLP_2026_SEMESTER_EXER.git FAC-Synthesis
  echo "✅ Repo cloned successfully."
else
  echo "ℹ️ FAC-Synthesis already exists, skipping clone."
fi

## 4. Patch για 4-bit Llama Loading

Το Llama 3.1 8B κανονικά απαιτεί 16GB VRAM. Για να αποφύγουμε Out Of Memory σφάλματα ή CPU offloading σε T4 GPU, πατσάρουμε το `llama_wrapper.py` για 4-bit precision configuration (`BitsAndBytesConfig`), μειώνοντας τις απαιτήσεις VRAM σε μόλις ~6GB.

In [ ]:
import os

wrapper_path = "/content/FAC-Synthesis/fac_synthesis/step2_feature_covered_sample_synthesis/llama_wrapper.py"

with open(wrapper_path, "r") as f:
    code = f.read()

patch = """from transformers import BitsAndBytesConfig
import torch
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)"""

code = code.replace("""model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)""", patch)

with open(wrapper_path, "w") as f:
    f.write(code)

print("✅ Το llama_wrapper.py (step2) ενημερώθηκε επιτυχώς για 4-bit precision!")

## 5. Αντιγραφή του JSONL Input (Contrastive Pairs)

**Διαφορά από Phase 4a:** Εκεί η είσοδος ήταν ένα TSV αρχείο (`missing_features.tsv`). Εδώ η είσοδος είναι ένα **JSONL** αρχείο, με ένα JSON object ανά γραμμή. Κάθε γραμμή περιέχει:

```json
{
  "feature_id": 1226,
  "Feature Summary": "...",
  "Good example": "query κείμενο που ενεργοποίησε ισχυρά το feature",
  "Good Span Activated": "τα tokens που πυροδότησαν το activation",
  "Good Activation score": 5.12,
  "Bad example": "query κείμενο με χαμηλότερο activation",
  "Bad Span Activated": "τα αντίστοιχα spans",
  "Bad Activation score": 0.83
}
```

Αυτά τα contrastive pairs προέκυψαν από τη διαδικασία 4a→4b (SAE scoring + analyze + merge).

**Αν δεν έχεις ακόμα αυτό το αρχείο**, πρέπει πρώτα να ολοκληρώσεις τα Phase 4a + 4b notebooks.

In [ ]:
%%bash
# ΠΡΟΣΟΧΗ: Άλλαξε το path ανάλογα με το πού αποθήκευσες το output του Phase 4b!
INPUT_JSONL="/content/drive/MyDrive/fac_synthesis/step_4/4b/step1_contrastive_pairs.jsonl"

if [ -f "$INPUT_JSONL" ]; then
  cp "$INPUT_JSONL" /content/contrastive_pairs.jsonl
  echo "✅ Αντιγράφηκε το JSONL input."
  wc -l /content/contrastive_pairs.jsonl
else
  echo "❌ Δεν βρέθηκε το αρχείο: $INPUT_JSONL"
  echo "   Σιγουρέψου ότι έχεις τρέξει πρώτα τα Phase 4a + 4b."
fi

### Προεπισκόπηση JSONL Input
Ας δούμε τι δομή έχει ένα entry, για να επιβεβαιώσουμε ότι φορτώθηκε σωστά.

In [ ]:
import json

with open("/content/contrastive_pairs.jsonl", "r") as f:
    first_entry = json.loads(f.readline())

print("Keys:", list(first_entry.keys()))
print()
for k, v in first_entry.items():
    v_str = str(v)
    print(f"  {k}: {v_str[:120]}{'...' if len(v_str) > 120 else ''}")

## 6. Εκτέλεση του Generation (Phase 4c — Round 2)

**Χαρακτηριστικά εκτέλεσης:**
- Τρέχουμε `generate_data_llama_r2.py` (αντί `generate_data_llama_r1.py`)
- Χρησιμοποιούμε το **Uncensored/Abliterated Llama model**
- Τo `--features` δέχεται JSONL (αντί TSV)
- Τo output αποθηκεύεται κατευθείαν στο Drive (για persistence)

### 🔄 RESUME SUPPORT (Υποστηρίζεται πλέον!)
Το `generate_data_llama_r2.py` έχει αναβαθμιστεί πλήρως για να υποστηρίζει **progress tracking** και **incremental disk writes**. Αν η εκτέλεση διακοπεί, θα συνεχίσει αυτόματα από το feature που σταμάτησε, χωρίς να χάσει τις ήδη παραχθείσες γραμμές!

In [ ]:
# Δημιουργία output folder στο Drive
!mkdir -p "/content/drive/MyDrive/fac_synthesis/step_4/4c/log_files"

! cd /content/FAC-Synthesis/fac_synthesis/step2_feature_covered_sample_synthesis/ && python generate_data_llama_r2.py \
  --features /content/contrastive_pairs.jsonl \
  --out "/content/drive/MyDrive/fac_synthesis/step_4/4c/log_files/step2_queries" \
  --temperature 0.8

## 7. Επαλήθευση Output

Ελέγχουμε ότι το TSV παράχθηκε σωστά και ρίχνουμε μια ματιά στα πρώτα queries.

In [ ]:
import os

tsv_path = "/content/drive/MyDrive/fac_synthesis/step_4/4c/log_files/step2_queries.queries.tsv"

if os.path.exists(tsv_path):
    with open(tsv_path, "r") as f:
        lines = f.readlines()
    print(f"✅ Σύνολο generated queries: {len(lines)}")
    print(f"\n--- Πρώτα 5 queries ---")
    for i, line in enumerate(lines[:5]):
        parts = line.strip().split("\t")
        query = parts[0] if parts else "???"
        label = parts[1] if len(parts) > 1 else "?"
        print(f"  [{i+1}] (label={label}) {query[:150]}{'...' if len(query) > 150 else ''}")
else:
    print(f"❌ Δεν βρέθηκε το output: {tsv_path}")

## 8. Καθαρισμός Trailing Artifacts

Ίδιο πρόβλημα με Phase 4a: η οδηγία `<TAB>1` στο prompt μπορεί να παράγει αλλοιωμένα suffixes όπως `Tab 1`, `(TAB) 1`, `\n1`, κλπ. Τα καθαρίζουμε αυτόματα.

In [ ]:
import re

tsv_path = "/content/drive/MyDrive/fac_synthesis/step_4/4c/log_files/step2_queries.queries.tsv"

def clean_query(q):
    # Remove prefix like Query-1: / Query 1: case-insensitively
    q = re.sub(r'Query[- ]\d+\s*:\s*', '', q, flags=re.I)
    # Remove Tab 1 / TAB 1 / (TAB) 1 / Tab: 1 variants at the end
    q = re.sub(r'\\n\(?TAB\)?\s*[:\-]?\s*1\s*$', '', q, flags=re.I)
    q = re.sub(r'\\nTab\s*[:\-]?\s*1\s*$', '', q, flags=re.I)
    q = re.sub(r'\\n\(?Tab\)?\s*[:\-]?\s*1\s*$', '', q, flags=re.I)
    # Remove raw \n1 at the end
    q = re.sub(r'\\n1\s*$', '', q)
    # Remove space + 1 at the end (e.g. "? 1" -> "?")
    q = re.sub(r'(?<=[a-zA-Z\.\?])\s+1\s*$', '', q)
    # Remove trailing \t0 that the script appends when LLM didn't emit its own label
    q = re.sub(r'\s*\\t0\s*$', '', q)
    # Clean any internal escaped or raw newlines, double spaces
    q = q.replace('\\n', ' ').replace('\n', ' ').replace('\r', '')
    q = re.sub(r'\s+', ' ', q)
    return q.strip()

with open(tsv_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

cleaned_lines = []
changes = 0
for line in lines:
    line_stripped = line.strip('\n')
    if not line_stripped or '\t' not in line_stripped:
        cleaned_lines.append(line)
        continue
    query_part, label_part = line_stripped.rsplit('\t', 1)
    cleaned_query = clean_query(query_part)
    if cleaned_query != query_part:
        changes += 1
    cleaned_lines.append(f"{cleaned_query}\t{label_part}\n")

with open(tsv_path, "w", encoding="utf-8") as f:
    f.writelines(cleaned_lines)

print(f"✅ Καθαρισμός ολοκληρώθηκε! Αλλαγές: {changes}/{len(lines)} γραμμές")


## 9. Αποθήκευση στο Google Drive

Τα αποτελέσματα γράφονται ήδη κατευθείαν στο Drive (`/content/drive/MyDrive/fac_synthesis/step_4/4c/log_files/`). Ας επιβεβαιώσουμε:

In [ ]:
%%bash
echo "📁 Αρχεία στο Drive output folder:"
ls -lh "/content/drive/MyDrive/fac_synthesis/step_4/4c/log_files/"
echo ""
echo "✅ ΤΕΛΟΣ Phase 4c! Τα Round 2 queries είναι αποθηκευμένα στο Drive."

---

## Σύνοψη Pipeline

```
Phase 4a (Round 1)          Phase 4b (SAE Scoring)         Phase 4c (Round 2)
┌─────────────────┐        ┌─────────────────────┐        ┌──────────────────┐
│ missing_features │───────►│ collect_spans.py    │        │ generate_data    │
│     .tsv         │        │ analyze_step1_...py │        │  _llama_r2.py    │
│                  │        │ merge_step1_...py   │        │                  │
│ generate_data    │        │                     │        │ Input: JSONL     │
│  _llama_r1.py    │        │ Output: contrastive │───────►│  (contrastive    │
│                  │        │  _pairs.jsonl       │        │   pairs + SAE    │
│ Output: queries  │───────►│                     │        │   scores)        │
│  .queries.tsv    │        └─────────────────────┘        │                  │
└─────────────────┘                                        │ Output: refined  │
                                                           │  queries.tsv     │
                                                           └──────────────────┘
```

Τα Round 2 queries μπορούν πλέον να χρησιμοποιηθούν για fine-tuning ή για έναν δεύτερο κύκλο SAE scoring.